In [ ]:
# =============================================================================
# jobs/32_build_responder_geographies.py        SKELETON: runs end to end for the geographies NSUL carries, with marked gaps
# The geographies responders work to, with names, and the flood exposure of each (plan: docs/EXPOSURE_AND_RISK_GRID_PLAN.md).
#
# WHAT IT DOES NOW
#   1. Finds the code-to-name reference files in the NSUL reference folder and reads a name for every code of: local authority district, county or
#      unitary authority, ward, constituency, police force area and region. Falls back to the ONS boundary files for districts and counties.
#      It prints what it found for each geography and what it could not.
#   2. Writes config.TBL_GEOGRAPHY_NAMES (geography, code, name, source). Names and codes are open data, so the unrestricted schema.
#   3. Adds up the address points by each geography's code, with the same measures as the warning areas: people, households, over-65s, addresses,
#      people by rivers and sea band and by Flood Zone, and critical sites. Writes config.TBL_GEOGRAPHY_EXPOSURE (restricted schema).
#
# WHAT IS MISSING, IN ORDER OF IMPORTANCE (each is marked TODO below)
#   a. Fire and rescue authorities and combined authorities. NSUL has neither. FIRE AND RESCUE: give the ONS boundary file (GeoJSON, "Fire and
#      Rescue Authorities (December 2024) Boundaries EW BFC") in fra_geojson. Each output area's centre is placed in an authority (the nearest within
#      2 km if it falls just outside a boundary clipped to the coast), and the address points are added up by output area. That is exact where the
#      authority boundary cuts through a district, which a district lookup cannot do. COMBINED AUTHORITIES: the ONS "Local Authority District to
#      Combined Authority (May 2025) Lookup in EN" CSV, in combined_authority_csv. Download the ONS lookups of local authority districts to each,
#      put them in a volume, and give their paths in the widgets combined_authority_csv and fire_authority_csv (the raw ONS files: they are
#      reshaped here). Check that the version matches the 2025 district codes (LAD25CD): the notebook reports any district that is in no row.
#      Or give one file already shaped as district_code, geography, geography_code, geography_name in lookup_csv.
#   b. Constituency and ward names if the NSUL reference folder has no file for them: use the ONS boundary files in the lake.
#   c. Local resilience forums: out of scope for now.
#   d. A suppressed version of the exposure table for the app: small wards can hold few people.
#
# Widgets: nsul_ref_dir, combined_authority_csv, fire_authority_csv, fra_geojson, oa_file, lookup_csv.
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

# the boundary-file step needs these; they are not on every cluster. Installed if missing, and Python restarted only if they still cannot be imported
def _have():
    try:
        import pyproj  # noqa: F401
        import shapely  # noqa: F401
        return True
    except ImportError:
        return False


if not _have():
    get_ipython().run_line_magic("pip", "install -q pyproj shapely")
    import importlib
    importlib.invalidate_caches()
    if not _have():
        print("Packages installed but not importable in this session: restarting Python. Run all again from the top.")
        dbutils.library.restartPython()

import importlib
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

import config
import importlib as _il

_il.reload(config)   # a Python session that ran an earlier version of config.py would otherwise miss tables added since
import utils.responder_geo as _rg
importlib.reload(_rg)
from utils.area_exposure import to_bng_polygons
from utils.responder_geo import (GEOGRAPHIES, LOOKUP_GEOGRAPHIES, assign_points_to_polygons, fill_missing_names, find_reference_file, parse_code_name, parse_district_lookup,
                                 read_csv_text, read_geojson_features, rollup_through_lookup)
from utils.helpers import get_spark

spark = get_spark()
ONS = "/Volumes/prd_dash_bronze/ons_open_geography_portal_unrestricted"


def show(df):
    display(df.fillna("").astype(str) if len(df) else df)


dbutils.widgets.text("nsul_ref_dir", f"{ONS}/national_statistics_uprn_lookup_ref/format_CSV_national_statistics_uprn_lookup_ref/LATEST_national_statistics_uprn_lookup_ref/", "NSUL reference folder")
dbutils.widgets.text("combined_authority_csv", "", "ONS lookup of local authority districts to combined authorities (the raw ONS CSV; optional)")
dbutils.widgets.text("fire_authority_csv", "", "ONS lookup of local authority districts to fire and rescue authorities (the raw ONS CSV; optional)")
dbutils.widgets.text("fra_geojson", "", "Fire and rescue authority boundaries, ONS GeoJSON (December 2024 EW BFC; optional)")
dbutils.widgets.text("oa_file", f"{ONS}/output_areas_2021_ew_bfe/format_GEOPARQUET_output_areas_2021_ew_bfe/LATEST_output_areas_2021_ew_bfe", "Output areas file (centres, to place each in a fire authority)")
dbutils.widgets.text("lookup_csv", "", "Or one file already in the form: district_code, geography, geography_code, geography_name (optional)")
REF_DIR, LOOKUP_CSV = dbutils.widgets.get("nsul_ref_dir").strip(), dbutils.widgets.get("lookup_csv").strip()
CA_CSV, FRA_CSV = dbutils.widgets.get("combined_authority_csv").strip(), dbutils.widgets.get("fire_authority_csv").strip()
FRA_GEOJSON, OA_FILE = dbutils.widgets.get("fra_geojson").strip(), dbutils.widgets.get("oa_file").strip()


In [ ]:
# =============================================================================
# 1. NAMES: the NSUL reference files, then the boundary files as a fallback
# =============================================================================

ref_files = [f.name for f in dbutils.fs.ls(REF_DIR)]
print(f"{len(ref_files)} files in the NSUL reference folder:")
show(pd.DataFrame({"file": ref_files}))

BOUNDARY_FALLBACK = {   # geography -> (volume folder, code column, name column)
    "local_authority_district": (f"{ONS}/local_authority_districts_bdy_uk_bfe/format_GEOPARQUET_local_authority_districts_bdy_uk_bfe/LATEST_local_authority_districts_bdy_uk_bfe", "LAD25CD", "LAD25NM"),
    "county_or_unitary": (f"{ONS}/counties_uni_authorities_bdy_uk_bfe/format_GEOPARQUET_counties_uni_authorities_bdy_uk_bfe/LATEST_counties_uni_authorities_bdy_uk_bfe", "CTYUA25CD", "CTYUA25NM"),
}

names, report = [], []
for geo, spec in GEOGRAPHIES.items():
    f = find_reference_file(ref_files, spec["keywords"])
    got, how = None, ""
    if f:
        try:
            df = read_csv_text(os.path.join(REF_DIR, f))
            got, code_col, name_col = parse_code_name(df)
            how = f"{f} ({code_col}, {name_col})" if got is not None else f"{f}: no code and name columns, found {list(df.columns)[:6]}"
        except Exception as e:
            how = f"{f}: could not read ({str(e)[:80]})"
    if got is None and geo in BOUNDARY_FALLBACK:
        folder, cc, nc = BOUNDARY_FALLBACK[geo]
        b = spark.read.parquet(folder).select(cc, nc).toPandas()
        got = pd.DataFrame({"code": b[cc].astype(str), "name": b[nc].astype(str)}).drop_duplicates("code")
        how = f"ONS boundary file ({cc}, {nc})"
    # TODO (b): constituency and ward names, if neither route above found them: the ONS boundary files in the lake (Wards, Westminster Parliamentary Constituencies)
    if got is not None:
        got = got.assign(geography=geo, source=how)
        names.append(got)
    report.append({"geography": geo, "names": 0 if got is None else len(got), "found_in": how or "NOT FOUND"})
print("Names found, by geography:")
show(pd.DataFrame(report))
names_df = pd.concat(names, ignore_index=True)[["geography", "code", "name", "source"]] if names else pd.DataFrame(columns=["geography", "code", "name", "source"])

# The county file names counties only; unitary and metropolitan authority codes sit in the same column. Fill them from the boundary file, then the district names.
_b = BOUNDARY_FALLBACK["county_or_unitary"]
_bdy = spark.read.parquet(_b[0]).select(_b[1], _b[2]).toPandas().rename(columns={_b[1]: "code", _b[2]: "name"})
_lad = names_df[names_df["geography"] == "local_authority_district"][["code", "name"]]
_before = int((names_df["geography"] == "county_or_unitary").sum())
names_df = fill_missing_names(names_df, "county_or_unitary", [(_bdy, "ONS boundary file (unitary fill)"), (_lad, "district names (unitary fill)")])
print(f"County or unitary names: {_before} from the reference file, {int((names_df['geography'] == 'county_or_unitary').sum())} after filling unitary and metropolitan codes.")


In [ ]:
# =============================================================================
# 2. EXPOSURE BY GEOGRAPHY, from the address points
# One Spark aggregation per geography: no tile loop is needed, because it is only a sum by code.
# =============================================================================

points = spark.table(config.TBL_ADDRESS_POINTS)
agg = [F.sum(F.when(F.col("point_source") == "nsul", 1).otherwise(0)).alias("addresses"),
       F.sum("people").alias("people"), F.sum("households").alias("households"), F.sum("over65").alias("over65"),
       F.sum(F.when(F.col("rofrs_rank") == 1, F.col("people")).otherwise(0)).alias("people_rs_very_low"),
       F.sum(F.when(F.col("rofrs_rank") == 2, F.col("people")).otherwise(0)).alias("people_rs_low"),
       F.sum(F.when(F.col("rofrs_rank") == 3, F.col("people")).otherwise(0)).alias("people_rs_medium"),
       F.sum(F.when(F.col("rofrs_rank") == 4, F.col("people")).otherwise(0)).alias("people_rs_high"),
       F.sum(F.when(F.col("rofrs_rank") >= 3, F.col("people")).otherwise(0)).alias("people_rs_high_medium"),
       F.sum(F.when(F.col("rofrs_rank") >= 3, F.col("households")).otherwise(0)).alias("households_rs_high_medium"),
       F.sum(F.when(F.col("fz_rank") == 1, F.col("people")).otherwise(0)).alias("people_fz2"),
       F.sum(F.when(F.col("fz_rank") == 2, F.col("people")).otherwise(0)).alias("people_fz3"),
       F.sum(F.when(F.col("fz_sea_rank") == 2, F.col("people")).otherwise(0)).alias("people_fz3_sea"),
       F.sum(F.when(F.col("fz_river_rank") == 2, F.col("people")).otherwise(0)).alias("people_fz3_river"),
       F.sum(F.when(F.col("fz_river_and_sea_rank") == 2, F.col("people")).otherwise(0)).alias("people_fz3_both")]
VALUES = ["addresses", "people", "households", "over65", "people_rs_very_low", "people_rs_low", "people_rs_medium", "people_rs_high", "people_rs_high_medium",
          "households_rs_high_medium", "people_fz2", "people_fz3", "people_fz3_sea", "people_fz3_river", "people_fz3_both"]

parts = []
for geo, spec in GEOGRAPHIES.items():
    g = points.filter(F.col(spec["points_col"]).isNotNull()).groupBy(F.col(spec["points_col"]).alias("code")).agg(*agg).toPandas()
    g.insert(0, "geography", geo)
    parts.append(g)
    print(f"{geo}: {len(g):,} codes, {g['people'].sum():,.0f} people")
exposure = pd.concat(parts, ignore_index=True)

# critical sites per geography, if job 31 has run (the sites table has the local authority district only: TODO, add the other codes to the sites table)
if spark.catalog.tableExists(config.TBL_CRITICAL_SITES):
    s = (spark.table(config.TBL_CRITICAL_SITES).groupBy(F.col("lad").alias("code"))
         .agg(F.count("*").alias("sites"), F.sum(F.when(F.col("rofrs_rank") >= 3, 1).otherwise(0)).alias("sites_rs_high_medium")).toPandas())
    s["geography"] = "local_authority_district"
    exposure = exposure.merge(s, on=["geography", "code"], how="left")
for c in ("sites", "sites_rs_high_medium"):
    exposure[c] = exposure[c].fillna(0.0) if c in exposure else 0.0

# fire and rescue authorities from their boundary polygons: place each output area's centre in an authority, then add the address points up by output area
done_geos = set()
if FRA_GEOJSON:
    feats, epsg = read_geojson_features(open(FRA_GEOJSON, encoding="utf-8-sig").read())
    fra_polys, fra_status = to_bng_polygons(feats["geometry"].tolist(), source_epsg=epsg)
    keep = [i for i, s_ in enumerate(fra_status) if s_ == "ok"]
    cd_col = next(c for c in feats.columns if c != "geometry" and str(c).upper().endswith("CD"))
    nm_col = next(c for c in feats.columns if c != "geometry" and str(c).upper().endswith("NM"))
    fra = pd.DataFrame({"code": feats[cd_col].astype(str).to_numpy()[keep], "name": feats[nm_col].astype(str).to_numpy()[keep]})
    fra_polys = [fra_polys[i] for i in keep]
    print(f"Fire and rescue authorities: {len(fra)} polygons read ({len(feats) - len(keep)} features dropped) from {FRA_GEOJSON}.")
    oa_c = (spark.read.parquet(OA_FILE).select("OA21CD", "BNG_E", "BNG_N").toPandas())
    oa_c = oa_c[oa_c["OA21CD"].astype(str).str.startswith("E")].reset_index(drop=True)
    idx, n_near = assign_points_to_polygons(oa_c["BNG_E"].astype(float), oa_c["BNG_N"].astype(float), fra_polys)
    oa_c["fra_code"] = np.where(idx >= 0, fra["code"].to_numpy()[np.clip(idx, 0, None)], None)
    print(f"{len(oa_c):,} England output areas: {int((idx >= 0).sum()):,} placed ({n_near:,} by the nearest-boundary fallback), {int((idx < 0).sum()):,} in none.")
    by_oa = points.filter(F.col("oa").isNotNull()).groupBy(F.col("oa").alias("code")).agg(*agg).toPandas()
    m = by_oa.merge(oa_c[["OA21CD", "fra_code"]].rename(columns={"OA21CD": "code"}), on="code", how="left")
    print(f"People in output areas with no fire authority: {m.loc[m['fra_code'].isna(), 'people'].sum():,.0f} of {m['people'].sum():,.0f}.")
    fra_x = m[m["fra_code"].notna()].groupby("fra_code")[VALUES].sum().reset_index().rename(columns={"fra_code": "code"})
    fra_x.insert(0, "geography", "fire_and_rescue_authority")
    if spark.catalog.tableExists(config.TBL_CRITICAL_SITES):
        st_pts = spark.table(config.TBL_CRITICAL_SITES).select("easting", "northing", "rofrs_rank").toPandas()
        si, _ = assign_points_to_polygons(st_pts["easting"].astype(float), st_pts["northing"].astype(float), fra_polys, max_nearest_m=2000.0)
        st_pts["code"] = np.where(si >= 0, fra["code"].to_numpy()[np.clip(si, 0, None)], None)
        sc = st_pts[st_pts["code"].notna()].groupby("code").agg(sites=("code", "size"), sites_rs_high_medium=("rofrs_rank", lambda r: int((r >= 3).sum()))).reset_index()
        fra_x = fra_x.merge(sc, on="code", how="left")
    for c in ("sites", "sites_rs_high_medium"):
        fra_x[c] = fra_x[c].fillna(0.0) if c in fra_x else 0.0
    exposure = pd.concat([exposure, fra_x], ignore_index=True)
    names_df = pd.concat([names_df, fra.assign(geography="fire_and_rescue_authority", source="ONS boundary file (" + os.path.basename(FRA_GEOJSON) + ")")[["geography", "code", "name", "source"]]], ignore_index=True)
    done_geos.add("fire_and_rescue_authority")
    print(f"Fire and rescue authorities with people: {len(fra_x)}; people {fra_x['people'].sum():,.0f}.")

# TODO (a): combined authorities (and fire and rescue, if no boundary file was given), rolled up from the districts through lookup files
gaps = [g for g in LOOKUP_GEOGRAPHIES]
lookups = []
for path, geo in ((CA_CSV, "combined_authority"), (FRA_CSV, "fire_and_rescue_authority")):
    if path:
        lookups.append(parse_district_lookup(read_csv_text(path), geo))
if LOOKUP_CSV:
    lk0 = read_csv_text(LOOKUP_CSV)
    need = {"district_code", "geography", "geography_code", "geography_name"}
    if not need <= set(lk0.columns):
        raise ValueError(f"The lookup file needs the columns {sorted(need)}; it has {list(lk0.columns)}.")
    lookups.append(lk0[list(need)])
if lookups:
    lk = pd.concat(lookups, ignore_index=True)
    districts = exposure[exposure["geography"] == "local_authority_district"].drop(columns="geography")
    england_districts = districts[districts["code"].astype(str).str.startswith("E")]
    for geo in sorted(lk["geography"].unique()):
        unmatched = sorted(set(england_districts["code"]) - set(lk.loc[lk["geography"] == geo, "district_code"]))
        print(f"{geo}: {lk['geography'].eq(geo).sum():,} lookup rows; {len(unmatched)} English districts are in none" + (f" (for example {unmatched[:6]})" if unmatched else "") + ".")
    rolled = rollup_through_lookup(districts, lk, VALUES + ["sites", "sites_rs_high_medium"])
    exposure = pd.concat([exposure, rolled.rename(columns={"geography_code": "code"}).drop(columns=["districts"])[["geography", "code"] + VALUES + ["sites", "sites_rs_high_medium"]]], ignore_index=True)
    names_df = pd.concat([names_df, rolled.rename(columns={"geography_code": "code", "geography_name": "name"}).assign(source="lookup file")[["geography", "code", "name", "source"]]], ignore_index=True)
    done_geos |= set(rolled["geography"])
gaps = [g for g in LOOKUP_GEOGRAPHIES if g not in done_geos]
exposure = exposure.merge(names_df[["geography", "code", "name"]], on=["geography", "code"], how="left")
exposure["share_people_high_medium"] = (exposure["people_rs_high_medium"] / exposure["people"].where(exposure["people"] > 0)).round(4)
print(f"\nNot yet available: {gaps or 'nothing'}. Local resilience forums are out of scope.")


In [ ]:
# =============================================================================
# 3. WRITE AND CHECK
# =============================================================================

def write(pdf, table, comment):
    out = pdf.copy()
    for c in out.columns:
        if out[c].dtype == object:
            out[c] = out[c].map(lambda v: None if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))
    spark.createDataFrame(out).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    spark.sql(f"COMMENT ON TABLE {table} IS '{comment}'")
    print(f"Written {len(pdf):,} rows to {table}.")


write(names_df, config.TBL_GEOGRAPHY_NAMES, "Names for the geographies responders work to (local authority district, county or unitary authority, ward, constituency, police force area, region, and fire and rescue or combined authority where a lookup was supplied), from the NSUL reference files, the ONS boundary files and a supplied lookup. Job 32.")
write(exposure, config.TBL_GEOGRAPHY_EXPOSURE, "Flood exposure of each responder geography, counted from the address points: people, households, over-65s, people by rivers and sea band and Flood Zone (coastal, fluvial, mapped together) and critical sites. Rivers and sea only, no surface water. Restricted. Job 32.")

print("\nEvery geography should add up to the same national people (the Census total, 56,490,284):")
tot = exposure[exposure["geography"].isin(GEOGRAPHIES)].groupby("geography")["people"].sum().round(0).reset_index()
show(tot)
print("\nNames missing for codes that have people (the app would show a bare code):")
miss = exposure[exposure["name"].isna() & exposure["geography"].isin(GEOGRAPHIES)].groupby("geography").agg(codes=("code", "size"), people=("people", "sum")).reset_index()
show(miss if len(miss) else pd.DataFrame({"result": ["none"]}))
print("\nThe 15 local authority districts with the most people in the rivers and sea High or Medium band:")
show(exposure[exposure["geography"] == "local_authority_district"].sort_values("people_rs_high_medium", ascending=False).head(15)[["code", "name", "people", "people_rs_high_medium", "share_people_high_medium", "sites_rs_high_medium"]].round(3))
print("\nPolice force areas:")
show(exposure[exposure["geography"] == "police_force_area"].sort_values("people_rs_high_medium", ascending=False)[["code", "name", "people", "people_rs_high_medium", "share_people_high_medium"]].round(3))


In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success")
